In [6]:
from google.colab import files
files.upload()

Saving archive.zip to archive.zip


In [2]:
import zipfile

zip_path = "/content/archive.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Extraction terminée")

✅ Extraction terminée


In [3]:
import os

print(os.listdir("/content/dataset"))

['archive1']


In [4]:
print(os.listdir("/content/dataset/archive1"))

['seg_test', 'seg_pred', 'seg_train']


In [5]:
print(os.listdir("/content/dataset/archive1/seg_train/seg_train"))

['sea', 'forest', 'glacier', 'mountain', 'street', 'buildings']


In [6]:
train_dir = "/content/dataset/archive1/seg_train/seg_train"
test_dir = "/content/dataset/archive1/seg_test/seg_test"

In [7]:
import tensorflow as tf

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224, 224),
    batch_size=32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224, 224),
    batch_size=16
)

Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.


In [8]:
print("Classes :", train_ds.class_names)

Classes : ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']


In [9]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.shuffle(1000).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [10]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

In [11]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Input(shape=(224,224,3)),

    layers.Conv2D(96, (7,7), strides=2, activation='relu'),
    layers.MaxPooling2D((3,3), strides=2),

    layers.Conv2D(256, (5,5), padding='same', activation='relu'),
    layers.MaxPooling2D((3,3), strides=2),

    layers.Conv2D(384, (3,3), padding='same', activation='relu'),
    layers.Conv2D(384, (3,3), padding='same', activation='relu'),
    layers.Conv2D(256, (3,3), padding='same', activation='relu'),

    layers.MaxPooling2D((3,3), strides=2),

    layers.Flatten(),

    layers.Dense(1024, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(6, activation='softmax')
])

In [12]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [13]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

In [14]:
history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=5
)

Epoch 1/5
 49/439 ━━━━━━━━━━━━━━━━━━━━ 1:10:28 11s/step - accuracy: 0.1918 - loss: 2.9704

KeyboardInterrupt: 